# ChaoSSM — Phase 1c: LR-Schedule Diagnostic (decisive test for parity)

## Why this phase, and why it's the last one on parity

Phase 1b ruled out the sparse-credit-assignment hypothesis: curriculum barely moved anything
(`chaossm_corridor` +0.016 at p=0.0625; `linear` and `chaossm_free` showed no effect at all,
p=0.625 and 1.0), and no arm cleared the 0.55 learnability bar under either curriculum setting.

More importantly, the decay-from-peak pattern is **identical across every arm and every
curriculum setting**: mean best accuracy 0.656-0.694, mean settled accuracy 0.492-0.515, mean
decay 0.159-0.189 -- no meaningful spread anywhere. `linear` decays exactly as much as the
chaotic arms. That rules out chaos-specific instability and curriculum-specific instability
alike. Every arm finds a solution good enough to reach 64-73% accuracy at some point during
training, then loses it, regardless of architecture or difficulty schedule.

That is the signature of **a generic optimization instability, not an architectural or
task-design problem**: constant `lr=3e-4` for 2500 steps with no decay, on a task whose
solution requires a genuinely nonlinear interaction between two widely-separated bits, computed
before a purely linear readout -- plausibly a narrow minimum that continued noisy updates
disrupt once found.

## The test, and the stopping rule

Add a cosine learning-rate decay (`lr` from `3e-4` to ~0 over training) and check whether any
arm **retains** its peak instead of drifting off it. `curriculum` is dropped entirely -- 1b
showed it doesn't matter -- so this is a clean, minimal, single-variable test: 3 arms x 5 seeds
= 15 runs, same step budget as 1b.

**Stopping rule, decided in advance:** if no arm's mean settled accuracy clears 0.55 here
either, parity is reported as an honest limitation of this architecture family and training
recipe at this scale -- not chased further. Three sub-phases (1, 1b, 1c) is the budget allocated
to this question; the copy win and the recall reversal are already a complete, interesting
result on their own.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon
from torch.func import jacrev

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase1c"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase1c


## Configuration

`lr_schedule="cosine"` decays from `lr` to `lr_min` over `steps`. `lr_schedule="none"`
reproduces 1b's no-curriculum condition exactly, as a same-run control. Full difficulty
throughout, since curriculum is now settled as not the active variable.

In [2]:
SMOKE_TEST = False   # full run: 3 arms x {cosine, none} x 5 seeds

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    steps: int = 2500
    batch: int = 64
    lr: float = 3e-4
    lr_min: float = 1e-6
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    parity_len: int = 48
    min_gap: int = 4
    arms: tuple = ("linear", "chaossm_free", "chaossm_corridor")
    lr_schedules: tuple = ("cosine", "none")
    seeds: int = 5
    eval_every: int = 20
    learnability_bar: float = 0.55

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 200; CFG.seeds = 1
    print("*** SMOKE TEST -- pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "steps": "2500",
  "batch": "64",
  "lr": "0.0003",
  "lr_min": "1e-06",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "parity_len": "48",
  "min_gap": "4",
  "arms": "('linear', 'chaossm_free', 'chaossm_corridor')",
  "lr_schedules": "('cosine', 'none')",
  "seeds": "5",
  "eval_every": "20",
  "learnability_bar": "0.55"
}


## 1. Model -- unchanged from Phase 1 / 1b

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        return torch.zeros(b, self.N, device=dev) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev)

class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, phi_kind, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, phi_kind)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Estimators -- unchanged

In [4]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

## 3. Parity task -- full difficulty throughout (curriculum settled as a non-factor in 1b)

In [5]:
def make_parity_batch(cfg, batch, device, gen):
    L, V = cfg.parity_len, cfg.vocab
    seq = torch.randint(2, V, (batch, L), device=device, generator=gen)
    bit_a = torch.randint(0, 2, (batch,), device=device, generator=gen)
    bit_b = torch.randint(0, 2, (batch,), device=device, generator=gen)
    pos_a = torch.randint(0, L//3, (batch,), device=device, generator=gen)
    pos_b = torch.randint(2*L//3, L, (batch,), device=device, generator=gen)
    bidx = torch.arange(batch, device=device)
    seq[bidx, pos_a] = bit_a
    seq[bidx, pos_b] = bit_b
    tgt = torch.full((batch, L), -100, device=device)
    tgt[:, -1] = (bit_a ^ bit_b)
    return seq, tgt

VOCAB_SIZE, READOUT_SIZE = CFG.vocab, 2

## 4. Trainer with optional cosine LR schedule

In [6]:
def train_one(arm, lr_schedule, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    model = ChaoSSMLM(cfg, phi_kind, VOCAB_SIZE, READOUT_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.steps, eta_min=cfg.lr_min) \
            if lr_schedule == "cosine" else None

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    hist, diverged = [], False

    for step in range(cfg.steps):
        inp, tgt = make_parity_batch(cfg, cfg.batch, DEVICE, gen)
        need_traj = (arm != "linear")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE); lam_val = float("nan")
        if arm != "linear":
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()
        if sched is not None: sched.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % cfg.eval_every == 0:
            with torch.no_grad():
                mask = tgt != -100
                pred = logits.argmax(-1)
                acc = ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()
            cur_lr = opt.param_groups[0]["lr"]
            hist.append(dict(step=step, arm=arm, lr_schedule=lr_schedule, seed=seed,
                             loss=L_task.item(), acc=acc, lam=lam_val, gnorm=gn, lr=cur_lr))

    tail = hist[-max(1, len(hist)//5):]
    s = dict(arm=arm, lr_schedule=lr_schedule, seed=seed, diverged=diverged,
             final_acc=float(np.mean([h["acc"] for h in tail])) if tail else np.nan,
             best_acc=float(np.max([h["acc"] for h in hist])) if hist else np.nan,
             decay_from_peak=float(np.max([h["acc"] for h in hist]) -
                                   np.mean([h["acc"] for h in tail])) if hist else np.nan,
             lam_settled=float(np.nanmean([h["lam"] for h in tail])) if tail else np.nan)
    return s, pd.DataFrame(hist)

## 5. Run -- 3 arms x {cosine, none} x 5 seeds

In [7]:
rows, hists = [], []
total = len(CFG.arms) * len(CFG.lr_schedules) * CFG.seeds
done = 0
for sched_name in CFG.lr_schedules:
    for arm in CFG.arms:
        for sd in range(CFG.seeds):
            s, h = train_one(arm, sched_name, sd, CFG)
            rows.append(s); hists.append(h); done += 1
            flag = " DIVERGED" if s["diverged"] else ""
            print(f"[{done}/{total}] lr={sched_name:7s} {arm:18s} seed={sd}  "
                  f"final={s['final_acc']:.3f} (chance 0.500, bar {CFG.learnability_bar})  "
                  f"best={s['best_acc']:.3f}  decay={s['decay_from_peak']:.3f}  "
                  f"lam={s['lam_settled']:+.3f}{flag}")

SUM = pd.DataFrame(rows); HIST = pd.concat(hists, ignore_index=True)
savetable(SUM, "phase1c_summary.csv"); savetable(HIST, "phase1c_history.csv")

[1/30] lr=cosine  linear             seed=0  final=0.498 (chance 0.500, bar 0.55)  best=0.625  decay=0.127  lam=+nan
[2/30] lr=cosine  linear             seed=1  final=0.504 (chance 0.500, bar 0.55)  best=0.672  decay=0.167  lam=+nan
[3/30] lr=cosine  linear             seed=2  final=0.524 (chance 0.500, bar 0.55)  best=0.672  decay=0.148  lam=+nan
[4/30] lr=cosine  linear             seed=3  final=0.462 (chance 0.500, bar 0.55)  best=0.656  decay=0.194  lam=+nan
[5/30] lr=cosine  linear             seed=4  final=0.511 (chance 0.500, bar 0.55)  best=0.688  decay=0.176  lam=+nan
[6/30] lr=cosine  chaossm_free       seed=0  final=0.502 (chance 0.500, bar 0.55)  best=0.656  decay=0.154  lam=+0.324
[7/30] lr=cosine  chaossm_free       seed=1  final=0.516 (chance 0.500, bar 0.55)  best=0.641  decay=0.125  lam=+0.334
[8/30] lr=cosine  chaossm_free       seed=2  final=0.498 (chance 0.500, bar 0.55)  best=0.641  decay=0.143  lam=+0.258
[9/30] lr=cosine  chaossm_free       seed=3  final=0.504 (

## 6. Results -- did the schedule fix retention?

In [8]:
piv = SUM.pivot_table(index="arm", columns="lr_schedule", values="final_acc", aggfunc="mean")
decay_piv = SUM.pivot_table(index="arm", columns="lr_schedule", values="decay_from_peak", aggfunc="mean")
print("mean final accuracy (chance=0.500, learnability bar=0.55):")
print(piv.round(4).to_string())
print("\nmean decay from peak:")
print(decay_piv.round(4).to_string())
savetable(piv.reset_index(), "phase1c_results_table.csv")
savetable(decay_piv.reset_index(), "phase1c_decay_table.csv")

def paired(mask_a, mask_b, col="final_acc"):
    x = SUM[mask_a].sort_values("seed")[col].values
    y = SUM[mask_b].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(n=len(x), p=np.nan, median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(n=len(x), p=float(p), median_diff=float(np.median(x-y)))

print("\ncosine vs none, within each arm (final_acc):")
rows = []
for arm in CFG.arms:
    r = paired(SUM.arm.eq(arm) & SUM.lr_schedule.eq("cosine"), SUM.arm.eq(arm) & SUM.lr_schedule.eq("none"))
    r["arm"] = arm; rows.append(r)
    print(f"  {arm:18s} n={r['n']} p={r['p']:.4f} median_diff={r['median_diff']:+.4f}")
ST = pd.DataFrame(rows); savetable(ST, "phase1c_schedule_effect.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

any_learned = bool(piv.max().max() > CFG.learnability_bar)
print(f"\nDECISION: any condition clears the {CFG.learnability_bar} learnability bar: {any_learned}")

mean final accuracy (chance=0.500, learnability bar=0.55):
lr_schedule       cosine    none
arm                             
chaossm_corridor  0.4979  0.4979
chaossm_free      0.5104  0.5028
linear            0.4999  0.5041

mean decay from peak:
lr_schedule       cosine    none
arm                             
chaossm_corridor  0.1521  0.1552
chaossm_free      0.1459  0.1598
linear            0.1626  0.1428
saved /content/drive/My Drive/ChaoSSM/results/phase1c/phase1c_results_table.csv
saved /content/drive/My Drive/ChaoSSM/results/phase1c/phase1c_decay_table.csv

cosine vs none, within each arm (final_acc):
  linear             n=5 p=1.0000 median_diff=-0.0031
  chaossm_free       n=5 p=0.3125 median_diff=+0.0188
  chaossm_corridor   n=5 p=1.0000 median_diff=+0.0013
saved /content/drive/My Drive/ChaoSSM/results/phase1c/phase1c_schedule_effect.csv

min achievable two-sided p at n=5: 0.0625

DECISION: any condition clears the 0.55 learnability bar: False


In [9]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for arm in CFG.arms:
    for sched_name, ls in [("cosine","-"), ("none","--")]:
        d = HIST[(HIST.arm==arm) & (HIST.lr_schedule==sched_name)].groupby("step").acc.mean()
        ax[0].plot(d.index, d.values, ls, label=f"{arm} ({sched_name})", alpha=.85)
ax[0].axhline(0.5, color="r", ls=":", label="chance")
ax[0].axhline(CFG.learnability_bar, color="orange", ls=":", label="learnability bar")
ax[0].set(xlabel="step", ylabel="parity accuracy", title="Does cosine decay stop the collapse?")
ax[0].legend(fontsize=6); ax[0].grid(alpha=.3)

d2 = HIST[(HIST.arm=="chaossm_free")].groupby(["step","lr_schedule"]).lr.mean().unstack()
ax[1].plot(d2.index, d2["cosine"], label="cosine")
ax[1].plot(d2.index, d2["none"], label="none")
ax[1].set(xlabel="step", ylabel="learning rate", title="LR schedule actually applied", yscale="log")
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase1c_results.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase1c/phase1c_results.png


## 7. Verdict -- and the stopping rule takes effect here

| Observation | Reading |
|---|---|
| Cosine schedule clears 0.55 on any arm, decay shrinks toward 0 | Confirmed: the collapse was a generic optimization-stability artifact, not an architecture or task-design limit. Report the schedule as part of the training recipe and re-run the Phase 1 chaos-vs-linear comparison on parity with it. |
| Cosine reduces decay but still doesn't clear 0.55 | Partial confirmation -- the instability hypothesis is directionally right but not sufficient alone; would need a second fix (e.g. smaller lr, more capacity) to fully resolve, which is beyond this phase's budget. |
| No difference between cosine and none | The instability hypothesis is wrong; something else is capping this task's learnability at this scale. Per the stopping rule, parity is reported as an honest limitation and this line of investigation ends here. |

Per the stopping rule decided before this run: if no condition clears 0.55, parity is written
up as a limitation -- this architecture family and training recipe cannot solve 48-token
sparse-signal parity at this scale -- and the project moves to writing up the Phase 1 results
that did work (the copy win and the associative-recall reversal), rather than a fourth parity
sub-phase.

In [10]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    results_table=piv.round(4).to_dict("index"),
    decay_table=decay_piv.round(4).to_dict("index"),
    schedule_effect=ST.to_dict("records"),
    learnability_bar=CFG.learnability_bar,
    any_learned=any_learned,
    decision="continue investigating parity" if any_learned else
             "STOP per pre-registered rule: report parity as a limitation, write up copy + recall results",
    config=dict(steps=CFG.steps, seeds=CFG.seeds, lr=CFG.lr, lr_min=CFG.lr_min),
)
savejson(verdict, "phase1c_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase1c/phase1c_verdict.json
{
  "smoke_test": false,
  "results_table": {
    "chaossm_corridor": {
      "cosine": 0.4979,
      "none": 0.4979
    },
    "chaossm_free": {
      "cosine": 0.5104,
      "none": 0.5028
    },
    "linear": {
      "cosine": 0.4999,
      "none": 0.5041
    }
  },
  "decay_table": {
    "chaossm_corridor": {
      "cosine": 0.1521,
      "none": 0.1552
    },
    "chaossm_free": {
      "cosine": 0.1459,
      "none": 0.1598
    },
    "linear": {
      "cosine": 0.1626,
      "none": 0.1428
    }
  },
  "schedule_effect": [
    {
      "n": 5,
      "p": 1.0,
      "median_diff": -0.003124999999999989,
      "arm": "linear"
    },
    {
      "n": 5,
      "p": 0.3125,
      "median_diff": 0.018750000000000044,
      "arm": "chaossm_free"
    },
    {
      "n": 5,
      "p": 1.0,
      "median_diff": 0.0012500000000000289,
      "arm": "chaossm_corridor"
    }
  ],
  "learnability_bar": 0.55,
  "any_learn